## Playing cards with Python

In this practise, we are going to explore the https://deckofcardsapi.com API. This API allows to create decks of cards, draw cards for them, shuffleling and pilling. All in all, it allows to play some card games while learning about APIs.

As the first step, lets explore the website, which includes the API documentation

In [ ]:
import requests

We define the base url of the API. All the endpoints have this base url, so we can use it as our base.

- The Python library to process urls is urllib.
- We can use the urlunsplit function to build an URL from its different elementes.

In [ ]:
from urllib.parse import urlunsplit, urlencode

SCHEME = "https"
BASE = "deckofcardsapi.com/api"
PATH = "/deck/new/"
QUERY = {"count": 1, "test": 2}
QUERY = urlencode(QUERY)

print(QUERY)

URL = urlunsplit((SCHEME, BASE, PATH, QUERY, ""))

print(URL)

In [ ]:
from urllib.parse import urlunsplit, urlencode, urljoin

SCHEME = "https"
BASE = "deckofcardsapi.com/api"


def build_api_url(path, query_dict=None):
    if query_dict is None:
        query_dict = {}
    query = urlencode(query_dict)
    return urlunsplit((SCHEME, BASE, path, query, ""))

In [ ]:
build_api_url("deck")

Now that we have learned how to build the URL, lets explore the API.

The first endpoint allows us to create a deck of cards. We are going to send a Post request to create a Deck of Cards. We choose Post because we are creating a new object. 

Note that in this case this API supports Post and Get for this task, but we have decided to stay in the common convention.

In [ ]:
path = "deck/new/"

request_url = build_api_url(path)
response = requests.post(url=request_url)

If we explore the response we see that we have received a Response object

In [ ]:
response

This object has many attributes like the error code, or the message.

In [ ]:
print("The status code is: ", response.status_code)
print("The response is: ", response.json())

In the response object we can also explore the request that we have made acessing the .request property

In [ ]:
response.request.headers

Coming back to our deck of cards, we have now create a deck of cards, lets observe the answer again

In [ ]:
response.json()

In [ ]:
deck_id = response.json()["deck_id"]

Now we are going to shuffle de cards. Some requests as we can see need a specific ID put in the path. This tells de API which deck we want to shuffle

In [ ]:
shuffle_path =  f"deck/{deck_id}/shuffle/"


In [ ]:
shuffle_path = f"deck/{deck_id}/shuffle/"

shuffle_url = build_api_url(shuffle_path)
shuffle_response = requests.post(url=shuffle_url)

print("The status code is: ", shuffle_response.status_code)
print("The response is: ", shuffle_response.json())

Enough of posting some stuff. Now lets start drawing cards. For that, we have the main draw method.

We see in the API docs that it draws a card from the Top of the deck. Lets see what is our first 3 cards.

To just retrieve stuff from an API, we should use the GET method.

The requests library allows to do this very easily too!

In [ ]:
draw_path = f"deck/{deck_id}/draw/"
query = {"count": 3}
draw_url = build_api_url(draw_path, query)

print(draw_url)

See how this time, we have passed a query parameter. With this parameter, we are telling the API we want 3 cards and not just 1 (which is the default)

In [ ]:
draw_response = requests.get(url=draw_url)

print("The status code is: ", draw_response.status_code)
print("The response is: ")
draw_response.json()

Now lets make a quick function to visualize the response better

In [ ]:
from requests import Response
from skimage import io as skio
import io
import matplotlib.pyplot as plt


def plot_cards(cards: list):
    n_cards = len(cards)
    fig, axes = plt.subplots(1, n_cards)
    for i in range(n_cards):
        b = requests.get(cards[i]["image"]).content
        image = skio.imread(io.BytesIO(b))
        axes[i].imshow(image)
        axes[i].axis("off")
    plt.show()

In [ ]:
plot_cards(draw_response.json()["cards"])

## Exercise 

1. Create a new deck
2. Shuffle it
3. Start drawing cards (only once at a time)
4. Put each suit (palo) in a different Pile.

In [ ]:
def create_deck(shuffled=True):
    if shuffled:
        path = "deck/new/shuffle/"
    else:
        path = "deck/new"
    request_url = build_api_url(path)
    response = requests.post(url=request_url)
    response.raise_for_status()
    return response.json()["deck_id"]



Observe how we can use the "raise_for_status" method to raise an error if our API doesnt return a successful code

In [ ]:
deck_id = create_deck()
print(deck_id)

In [ ]:
from requests import HTTPError

def draw_cards(deck_id: str, count=1):
    draw_path = f"deck/{deck_id}/draw/"
    query = {"count": count}
    draw_url = build_api_url(draw_path, query)
    draw_response = requests.get(url=draw_url)
    draw_response.raise_for_status()
    
    return draw_response.json()["cards"]

In [ ]:
def add_to_pile(deck_id: str, pile_name: str, card: dict):
    add_path = f"/deck/{deck_id}/pile/{pile_name}/add"
    query_dict = {"cards": card["code"]}
    pile_url = build_api_url(add_path, query_dict)
    pile_response = requests.get(url=pile_url)
    pile_response.raise_for_status()
    return pile_response.json()

def list_in_pile(deck_id: str, pile_name: str):
    list_path = f"/deck/{deck_id}/pile/{pile_name}/list"
    list_url = build_api_url(list_path)
    pile_response = requests.get(url=list_url)
    pile_response.raise_for_status()
    return pile_response.json()
    

In [ ]:
deck_id = create_deck(shuffled=True)
n_cards = 52

for i in range(n_cards):
    card = draw_cards(deck_id)[0]
    add_to_pile(deck_id, card["suit"], card)

piles_response = list_in_pile(deck_id, "HEARTS")

In [ ]:
plot_cards(piles_response["piles"]["HEARTS"]["cards"])